In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import cartesian_to_spherical


def dR(R, t):
    a = R[0]
    e = R[1]
    i = R[2]
    OMEGA = R[3]
    w = R[4]
    M = R[5]

    J2 = 1.08 * 10 ** (-3)
    Rt = 6378
    mu = 398600

    n = (mu / a ** 3) ** 0.5

    dadt = 0
    dedt = 0
    didt = 0
    dOMEGAdt = - (3 * n * J2 * (Rt ** 2) * np.cos(i)) / ((2 * (1 - e ** 2) ** 2) * a ** 2)
    dwdt = (3 / (4 * (1 - e ** 2) ** 2)) * (n * J2 * Rt ** 2) * ((5 * np.cos(i) ** 2 - 1) / a ** 2)
    dMdt = n + (3 / (4 * (1 - e ** 2) ** (3 / 2))) * (n * J2 * Rt ** 2) * ((3 * np.cos(i) ** 2 - 1) / a ** 2)

    return np.array([dadt, dedt, didt, dOMEGAdt, dwdt, dMdt])


def rungekutta(R, t, h):
    k1 = h * dR(R, t)
    k2 = h * dR(R + 0.5 * k1, t + 0.5 * h)
    k3 = h * dR(R + 0.5 * k2, t + 0.5 * h)
    k4 = h * dR(R + k3, t + h)

    R = R + (1 / 6) * (k1 + 2 * k2 + 2 * k3 + k4)
    return R


mu = 398600
# conditions initiales pour ISS
T = 24 * 60 * 60 / 15.48972323
n = (2 * np.pi) / T

# temps d'intégration
t_init = 366 * 24 * 3600
h = 1
tf = 3 * T + t_init

# initialise t1 et t_rot, qui serviront pour la suite
t1 = t_init
t_rot = 0

a0 = (mu / n ** 2) ** (1 / 3)
e0 = 0.004241
i0 = 51.6423 * np.pi / 180
w0 = 318.6083 * np.pi / 180
OMEGA0 = 162.1530 * np.pi / 180
M0 = 86.77624 * np.pi / 180

R = np.array([a0, e0, i0, OMEGA0, w0, M0])  # vecteur elements kepleriens

# declaration differentes listes pour contenir nos parametres x,y,z,temps,lattitude et longitude
X1 = []
Y1 = []
Z1 = []
Temps = []
THETA1 = []
PHI1 = []

# liste pour ranger tout les elements kepleriens
a1 = []
a2 = []
a3 = []
a4 = []
a5 = []
a6 = []

for t in np.arange(t_init, tf, h):

    # integration des elements kepleriens
    R = rungekutta(R, t, h)

    a = R[0]
    e = R[1]
    i = R[2]
    OMEGA = R[3]
    w = R[4]
    M = R[5]

    Temps.append(t)

    t1 = t1 + h
    if t1 - t_init > T:
        t1 = t_init  # permet de toujours rester entre 0 et 2 PI, car sinon les valeurs des anomalies depassent 2 PI,
        # bien qu'avec les proprietes du cos et sin ca ne changerai rien de les depasser

    M = M0 + (2 * np.pi / T) * (t1 - t_init)  # calcul de M en fonction du temps

    if M > 2 * np.pi:
        M = M - 2 * np.pi

    E = M
    for ii in range(0, 10, 1):
        E = E - (E - e * np.sin(E) - M) / (1 - e * np.cos(E))

    v = 2 * np.arctan(np.sqrt((1 + e) / (1 - e)) * np.tan(E / 2))
    if v < 0:
        v = v + 2 * np.pi

    # calcul parametres dans le plan de l ellipse
    r = a * (1 - e * e) / (1 + e * np.cos(v))
    P = r * np.cos(v)
    Q = r * np.sin(v)
    W = 0
    Vect = np.array([P, Q, W])  # vecteur en coord. cartesien dans le plan de l ellipse

    # initialisation matrices de rotation
    R_i = np.array([[1, 0, 0], [0, np.cos(i), -np.sin(i)], [0, np.sin(i), np.cos(i)]])
    R_w = np.array([[np.cos(w), -np.sin(w), 0], [np.sin(w), np.cos(w), 0], [0, 0, 1]])
    R_OMEGA = np.array([[np.cos(OMEGA), -np.sin(OMEGA), 0], [np.sin(OMEGA), np.cos(OMEGA), 0], [0, 0, 1]])

    Vect1 = np.dot(R_w, Vect)  # rotation around P-axis by -i         INCLINAISON
    Vect2 = np.dot(R_i, Vect1)  # rotation around W-axis by -w         PERIGEE ARGUMENT
    Vect3 = np.dot(R_OMEGA, Vect2)  # rotation around W-axis by -OMEGA     ASCENDING NODE LONGITUDE

    # vecteur coord. cartesien x,y,z dans le repere terrestre
    Rf = Vect3

    x = Rf[0]
    y = Rf[1]
    z = Rf[2]

    # extraction de x,y et z
    X1.append(x)
    Y1.append(y)
    Z1.append(z)

    # cartesian_to_spherical(x,y,z) permet de calculer les coord. spheriques en fonction des coord. cartesien x,y et z
    # extraction en degree de phi et theta
    coord_spheric = cartesian_to_spherical(x, y, z)
    theta = coord_spheric[1].degree
    phi = coord_spheric[2].degree

    # il faut prendre en compte la rotation de la Terre, on calcul donc son angle on dit alors que
    # son angle varie lineairement en fonction du temps, le ration t_rot/(3600*24) represente le
    # coefficient entre 0 et 1 qui calcul l angle, si il est superieur a 1, alors 1 jour est passé
    # et donc on le remet a zero
    t_rot = t_rot + h
    if t_rot / (24 * 3600) <= 1:
        angle_terre = t_rot * 360 / (24 * 3600)
    if t_rot / (24 * 3600) > 1:
        t_rot = 0
        angle_terre = 0

    # on soustrait l angle de la Terre a phi pour obtenir sa vraie trace au sol
    phi = phi - angle_terre

    # il peut y avoir quelque phi negatif du a la soustraction, du coup on rajoute 360 degree si phi est negatif
    if phi < 0:
        phi = phi + 360

    THETA1.append(theta)
    PHI1.append(phi)

    a1.append(a)
    a2.append(e)
    a3.append(i)
    a4.append(OMEGA)
    a5.append(w)
    a6.append(M)

# PLOT DES FIGURES

# on calcule ici la difference entre les points perturbee et non perturbee
# on soustrait alors les X1, Y1 et Z1 obtenues ici avec le vecteur x, y et z obtenues dans la partie precedente
dist = []

for i in range(len(X1)):
    dist.append(((x[i] - X1[i]) ** 2 + (y[i] - Y1[i]) ** 2 + (z[i] - Z1[i]) ** 2) ** 0.5)

plt.figure()
plt.title('Différence entre trajectoire perturbée et non-perturbée ')
plt.ylabel('différence [km]')
plt.xlabel('Temps [s]')
plt.plot(Temps, dist)

# on regarde ici l orbite perturbee

plt.figure(figsize=(8, 5))
ax = plt.axes(projection='3d')
ax.set_title('Trajectoire ISS perturbée')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_xlim3d(np.min(X1), np.max(X1))
ax.set_ylim3d(np.min(Y1), np.max(Y1))
ax.set_zlim3d(np.min(Z1), np.max(Z1))
ax.plot(X1, Y1, 'r')
plt.savefig("Trajectoire ISS perturbée")
ax.can_zoom()
plt.show()


TypeError: object of type 'numpy.float64' has no len()